In [1]:
from pydantic import BaseModel, Field
from typing import TypedDict, Literal

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_retries=5,    
)

In [3]:
class SentimentSchema(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Sentiment of the review')


class DiagnosisSchema(BaseModel):
    issue_type: Literal['UX', 'Performance', 'Bug', 'Support', 'Other'] = Field(
        description='The category of issue mentioned in the review'
    )
    tone: Literal['angry', 'frustrated', 'disappointed', 'calm'] = Field(
        description='The emotional tone expressed by the user'
    )
    urgency: Literal['low', 'medium', 'high'] = Field(
        description='How urgent or critical the issue appears to be'
    )

In [4]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)

In [5]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: dict
    response: str

In [7]:
def find_sentiment(state: ReviewState):
    prompt= f'for the following find out the sentiment \n {state['review']}'
    sentiment= structured_model.invoke(prompt).sentiment 
    return{'sentiment': sentiment}

In [8]:
def positive_response(state: ReviewState):
    prompt = f'''Write a warm thank-you message in response to this review:
\n\n{state["review"]}\n
Also, kindly ask the user to leave feedback on our website.'''
    response = model.invoke(prompt).content
    return {'response': response}

In [9]:
def run_diagnosis(state: ReviewState):
    prompt = f'''Diagnose this negative review:\n\n{state["review"]}\n
Return issue_type, tone, and urgency.'''
    response = structured_model2.invoke(prompt)
    return {'diagnosis': response.model_dump()}

In [10]:
def negative_response(state: ReviewState):
    diagnosis = state['diagnosis']
    prompt = f'''You are a support assistant.
The user had a '{diagnosis["issue_type"]}' issue, sounded '{diagnosis["tone"]}', and marked urgency as '{diagnosis["urgency"]}'.
Write an empathetic, helpful resolution message.'''
    response = model.invoke(prompt).content
    return {'response': response}

In [11]:
def check_sentiment(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [13]:
from langgraph.graph import StateGraph, START, END
from typing import Literal
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')

graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [14]:
positive_review = "I have been using this app for a month and the UI is incredibly clean and intuitive. Great job to the design team."
final_state = workflow.invoke({'review': positive_review})
print(final_state)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'review': 'I have been using this app for a month and the UI is incredibly clean and intuitive. Great job to the design team.', 'sentiment': 'positive', 'response': [{'type': 'text', 'text': 'Here is a warm and friendly response you can use:\n\n"Hi there! Thank you so much for the wonderful feedback! 🌟 We\'re thrilled to hear that you\'re enjoying the clean and intuitive design—I\'ll be sure to pass your kind words along to the design team; it will really make their day! \n\nSince you\'ve been using the app for a month now, we\'d love to hear more about your experience. If you have a quick moment, would you mind sharing a bit more feedback over on our website [insert link]? \n\nThanks again for your support and for being a part of our community!"', 'extras': {'signature': 'EmAKXgFpFH0TVReBIxSUU5WZufBs9RWcfpDpfF4ubnDh7R6IGsRBQSxjV5nbl73qyVvAiGyiQF40FvQ378mK//g/JUrlppQXzxfLAIYbudXKt7NcjHQFoVVPW19SrM4GZTo='}}]}


In [15]:
negative_review = "I have been trying to log in for over an hour and the app keeps freezing on the authentication screen. I even tried reinstalling. This kind of bug is unacceptable."
final_state = workflow.invoke({'review': negative_review})
print(final_state)

{'review': 'I have been trying to log in for over an hour and the app keeps freezing on the authentication screen. I even tried reinstalling. This kind of bug is unacceptable.', 'sentiment': 'negative', 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'}, 'response': [{'type': 'text', 'text': 'Subject: Apologies for the trouble — your bug fix update\n\nHi [User Name],\n\nI completely understand how frustrating it is to run into a bug, especially when you need things working smoothly right away. I’m really sorry for the disruption this has caused to your workflow. \n\nOur engineering team has treated this with high priority and has just released a fix for the issue you reported. \n\n**Here is what you need to do to apply the fix:**\n1. [Insert simple step, e.g., Refresh your browser / Restart the app / Clear your cache]\n2. [Insert verification step, e.g., Try performing the action again to ensure it works]\n\nCould you please verify on your end if everything is 